# Projeto Andromeda-Net: Visualização da Rede

Este notebook gera as visualizações finais do grafo Piloto–Construtor da Fórmula 1
usando `networkx` e `matplotlib`, seguindo as diretrizes de formatação do projeto:
- Paleta Seaborn consistente (`viridis` / `husl`)
- Layout Fruchterman-Reingold (`spring_layout`) para grafos densos
- Figuras exportadas em alta resolução (dpi=300) para o vídeo e README

> **Pré-requisito:** Execute antes `ETL/01_coleta_e_limpeza.ipynb` e `EDA/02_eda_e_redes.ipynb`.


In [ ]:
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from networkx.algorithms import bipartite

sns.set_theme(style="whitegrid")
FIGURES_DIR = "../../resultados/figuras/"
os.makedirs(FIGURES_DIR, exist_ok=True)

DATA_PATH = "../../data/processed/f1_graph_base.csv"
df = pd.read_csv(DATA_PATH)
print(f"Base carregada: {len(df)} registros")


## 1. Reconstrução do Grafo Bipartido

Recriamos o grafo Piloto–Construtor filtrado por pontuação (critério de "sucesso").
O peso da aresta é o total de pontos que o piloto marcou por aquela escuderia.


In [ ]:
# Filtrar apenas participações com pontuação > 0
df_pontos = df[df["points"] > 0].copy()

# Agrupar pontos totais por piloto–construtor
df_edges = df_pontos.groupby(["driver_name", "constructor_name"])["points"].sum().reset_index()

# Montar o grafo bipartido
B = nx.Graph()
drivers = df_edges["driver_name"].unique()
constructors = df_edges["constructor_name"].unique()
B.add_nodes_from(drivers, bipartite=0, node_type="driver")
B.add_nodes_from(constructors, bipartite=1, node_type="constructor")

for _, row in df_edges.iterrows():
    B.add_edge(row["driver_name"], row["constructor_name"], weight=row["points"])

print(f"Nós  : {B.number_of_nodes()}")
print(f"Arestas: {B.number_of_edges()}")
print(f"Bipartido válido? {bipartite.is_bipartite(B)}")


## 2. Visualização 1 — Subgrafo: Top 8 Construtoras e seus Pilotos

Para legibilidade, filtramos apenas as **8 construtoras com mais corridas pontuadas** e
os pilotos diretamente conectados a elas. Usamos `spring_layout` com semente fixa (seed=42).

- 🔵 **Nós azuis** → Construtoras (tamanho proporcional ao grau ponderado)
- 🟠 **Nós laranjas** → Pilotos (tamanho proporcional ao número de construtoras)


In [ ]:
# ── Selecionar top 8 construtoras por grau ponderado ──────────────────────
constructor_nodes = [n for n, d in B.nodes(data=True) if d["node_type"] == "constructor"]
constructor_degree = {n: B.degree(n, weight="weight") for n in constructor_nodes}
top_constructors = sorted(constructor_degree, key=constructor_degree.get, reverse=True)[:8]

# Sub-grafo: top construtoras + seus pilotos
sub_nodes = set(top_constructors)
for c in top_constructors:
    sub_nodes.update(B.neighbors(c))
SG = B.subgraph(sub_nodes)

# ── Layout ────────────────────────────────────────────────────────────────
pos = nx.spring_layout(SG, seed=42, k=0.6)

# ── Cores e tamanhos ──────────────────────────────────────────────────────
palette = sns.color_palette("husl", 8)   # uma cor por construtora
constructor_color_map = {c: palette[i] for i, c in enumerate(top_constructors)}

node_colors = []
node_sizes  = []
for n in SG.nodes():
    if SG.nodes[n]["node_type"] == "constructor":
        node_colors.append(constructor_color_map.get(n, "gray"))
        node_sizes.append(600 + constructor_degree.get(n, 0) * 0.1)
    else:
        node_colors.append("#F0A500")   # laranja para pilotos
        node_sizes.append(80)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 13))

# Arestas com espessura proporcional ao peso (pontos)
weights = [SG[u][v]["weight"] for u, v in SG.edges()]
max_w = max(weights)
edge_widths = [0.3 + 2.0 * (w / max_w) for w in weights]

nx.draw_networkx_edges(SG, pos, ax=ax, width=edge_widths, alpha=0.25, edge_color="gray")
nx.draw_networkx_nodes(SG, pos, ax=ax, node_color=node_colors, node_size=node_sizes, alpha=0.9)

# Rótulos apenas para as construtoras (evita poluição visual)
constructor_labels = {n: n for n in SG.nodes() if SG.nodes[n]["node_type"] == "constructor"}
nx.draw_networkx_labels(SG, pos, labels=constructor_labels, ax=ax, font_size=9, font_weight="bold")

# Legenda
legend_patches = [mpatches.Patch(color=constructor_color_map[c], label=c) for c in top_constructors]
legend_patches.append(mpatches.Patch(color="#F0A500", label="Pilotos"))
ax.legend(handles=legend_patches, loc="upper left", fontsize=8, title="Nós", title_fontsize=9)

ax.set_title("Subgrafo Bipartido: Top 8 Construtoras da F1 e seus Pilotos\n(espessura da aresta ∝ pontos conquistados)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "01_subgrafo_top_construtoras.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Figura salva em resultados/figuras/01_subgrafo_top_construtoras.png")


## 3. Visualização 2 — Distribuição de Grau dos Pilotos

Histograma do grau (número de construtoras distintas) de cada piloto na rede bipartida.
Redes com cauda longa (power-law) indicam poucos hubs muito conectados — típico de redes reais.


In [ ]:
driver_nodes = [n for n, d in B.nodes(data=True) if d["node_type"] == "driver"]
driver_degrees = [B.degree(n) for n in driver_nodes]

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(driver_degrees, bins=range(1, max(driver_degrees) + 2), color="#4C72B0",
             edgecolor="white", ax=ax)

ax.set_xlabel("Número de Construtoras por Piloto (Grau)", fontsize=12)
ax.set_ylabel("Número de Pilotos", fontsize=12)
ax.set_title("Distribuição de Grau dos Pilotos na Rede Bipartida\n(quantas escuderias distintas cada piloto marcou pontos)", fontsize=13)

# Linha de média
media = pd.Series(driver_degrees).mean()
ax.axvline(media, color="red", linestyle="--", label=f"Média = {media:.1f}")
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "02_distribuicao_grau_pilotos.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Figura salva em resultados/figuras/02_distribuicao_grau_pilotos.png")


## 4. Visualização 3 — Rede Projetada de Pilotos (Top Betweenness)

Projetamos o grafo bipartido sobre os pilotos e plotamos os **top 50 pilotos** com maior
Betweenness Centrality — os maiores "conectores" entre diferentes escuderias.

- **Tamanho do nó** ∝ Betweenness Centrality (maior = mais ponte)
- **Cor do nó** ∝ Eigenvector Centrality (mais quente = mais influente)


In [ ]:
# ── Projeção bipartida ────────────────────────────────────────────────────
driver_set = {n for n, d in B.nodes(data=True) if d["node_type"] == "driver"}
G_pilotos = bipartite.projected_graph(B, driver_set)

# ── Centralidades ─────────────────────────────────────────────────────────
print("Calculando centralidades (pode levar alguns segundos)...")
betweenness = nx.betweenness_centrality(G_pilotos)
eigenvector = nx.eigenvector_centrality(G_pilotos, max_iter=1000)

# Top 50 por betweenness
top50 = sorted(betweenness, key=betweenness.get, reverse=True)[:50]
SG2 = G_pilotos.subgraph(top50)

# ── Layout e atributos visuais ────────────────────────────────────────────
pos2 = nx.spring_layout(SG2, seed=42, k=1.5)

bet_vals = [betweenness[n] for n in SG2.nodes()]
eig_vals = [eigenvector[n] for n in SG2.nodes()]

# Normalizar tamanhos e cores
max_bet = max(bet_vals) if max(bet_vals) > 0 else 1
node_sizes2 = [200 + 4000 * (b / max_bet) for b in bet_vals]
norm_eig = [(e - min(eig_vals)) / (max(eig_vals) - min(eig_vals) + 1e-9) for e in eig_vals]

cmap = plt.cm.YlOrRd
node_colors2 = [cmap(v) for v in norm_eig]

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 14))

nx.draw_networkx_edges(SG2, pos2, ax=ax, alpha=0.15, edge_color="gray", width=0.5)
sc = nx.draw_networkx_nodes(SG2, pos2, ax=ax, node_color=norm_eig,
                             cmap=cmap, node_size=node_sizes2, alpha=0.85)
nx.draw_networkx_labels(SG2, pos2, ax=ax, font_size=6.5)

# Barra de cor para Eigenvector
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=min(eig_vals), vmax=max(eig_vals)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Eigenvector Centrality", fontsize=10)

ax.set_title(
    "Top 50 Pilotos por Betweenness Centrality — Rede Projetada\n"
    "(tamanho ∝ Betweenness | cor ∝ Eigenvector)",
    fontsize=14
)
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "03_projecao_pilotos_centralidade.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Figura salva em resultados/figuras/03_projecao_pilotos_centralidade.png")


## 5. Exportação da Tabela de Centralidade

Exportamos a tabela completa de métricas para `resultados/tabelas/centralidade_pilotos.csv`,
para ser referenciada no README e no vídeo.


In [ ]:
TABLES_DIR = "../../resultados/tabelas/"
os.makedirs(TABLES_DIR, exist_ok=True)

df_cent = pd.DataFrame({
    "Driver"      : list(betweenness.keys()),
    "Betweenness" : list(betweenness.values()),
    "Eigenvector" : [eigenvector[k] for k in betweenness],
    "Degree"      : [G_pilotos.degree(k) for k in betweenness],
}).sort_values("Betweenness", ascending=False).reset_index(drop=True)

df_cent.to_csv(os.path.join(TABLES_DIR, "centralidade_pilotos.csv"), index=False)
print("Top 10 pilotos por Betweenness Centrality:")
df_cent.head(10)
